# RNN人名分类案例

In [2]:
# 导入torch工具
import torch
# 导入nn准备构建模型
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
# 导入torch的数据源 数据迭代器工具包
from torch.utils.data import Dataset, DataLoader
# 用于获得常见字母及字符规范化
import string
# 导入时间工具包
import time
# 引入制图工具包
import matplotlib.pyplot as plt


# 获取所有常用字符包括字母和常用标点
all_letters = string.ascii_letters + " .,;'"
# 获取常用字符数量
n_letters = len(all_letters) # 57
print("n_letter:", n_letters)
# 国家名 种类数
categorys = ['Italian', 'English', 'Arabic', 'Spanish', 'Scottish', 'Irish', 'Chinese', 'Vietnamese', 'Japanese',
             'French', 'Greek', 'Dutch', 'Korean', 'Polish', 'Portuguese', 'Russian', 'Czech', 'German']
# 国家名 个数
categorynum = len(categorys)
print('categorys--->', categorys)# 国家名 种类数

n_letter: 57
categorys---> ['Italian', 'English', 'Arabic', 'Spanish', 'Scottish', 'Irish', 'Chinese', 'Vietnamese', 'Japanese', 'French', 'Greek', 'Dutch', 'Korean', 'Polish', 'Portuguese', 'Russian', 'Czech', 'German']


In [4]:
# 读取人名分类数据：数据读取到内存
def read_data(filename):
    my_list_x, my_list_y = [],[]

    with open(filename, mode="r", encoding="utf-8") as f:
        lines = f.readlines()
        print(lines)
        for line in lines:
            line = line.strip() # 去除左右空格
            if line:
                name, label = line.split("\t")
                my_list_x.append(name)
                my_list_y.append(label)
        print("my_list_x：",my_list_x)
        print("my_list_y：",my_list_y)

    return my_list_x, my_list_y


read_data("./assets/name_classfication.txt")

['Abl\tCzech\n', 'Adsit\tCzech\n', 'Ajdrna\tCzech\n', 'Alt\tCzech\n', 'Antonowitsch\tCzech\n', 'Antonowitz\tCzech\n', 'Bacon\tCzech\n', 'Ballalatak\tCzech\n', 'Ballaltick\tCzech\n', 'Bartonova\tCzech\n', 'Bastl\tCzech\n', 'Baroch\tCzech\n', 'Benesch\tCzech\n', 'Betlach\tCzech\n', 'Biganska\tCzech\n', 'Bilek\tCzech\n', 'Blahut\tCzech\n', 'Blazek\tCzech\n', 'Blazek\tCzech\n', 'Blazejovsky\tCzech\n', 'Blecha\tCzech\n', 'Bleskan\tCzech\n', 'Blober\tCzech\n', 'Bock\tCzech\n', 'Bohac\tCzech\n', 'Bohunovsky\tCzech\n', 'Bolcar\tCzech\n', 'Borovka\tCzech\n', 'Borovski\tCzech\n', 'Borowski\tCzech\n', 'Borovsky\tCzech\n', 'Brabbery\tCzech\n', 'Brezovjak\tCzech\n', 'Brousil\tCzech\n', 'Bruckner\tCzech\n', 'Buchta\tCzech\n', 'Cablikova\tCzech\n', 'Camfrlova\tCzech\n', 'Cap\tCzech\n', 'Cerda\tCzech\n', 'Cermak\tCzech\n', 'Chermak\tCzech\n', 'Cermak\tCzech\n', 'Cernochova\tCzech\n', 'Cernohous\tCzech\n', 'Cerny\tCzech\n', 'Cerney\tCzech\n', 'Cerny\tCzech\n', 'Cerv\tCzech\n', 'Cervenka\tCzech\n', 'Cha

(['Abl',
  'Adsit',
  'Ajdrna',
  'Alt',
  'Antonowitsch',
  'Antonowitz',
  'Bacon',
  'Ballalatak',
  'Ballaltick',
  'Bartonova',
  'Bastl',
  'Baroch',
  'Benesch',
  'Betlach',
  'Biganska',
  'Bilek',
  'Blahut',
  'Blazek',
  'Blazek',
  'Blazejovsky',
  'Blecha',
  'Bleskan',
  'Blober',
  'Bock',
  'Bohac',
  'Bohunovsky',
  'Bolcar',
  'Borovka',
  'Borovski',
  'Borowski',
  'Borovsky',
  'Brabbery',
  'Brezovjak',
  'Brousil',
  'Bruckner',
  'Buchta',
  'Cablikova',
  'Camfrlova',
  'Cap',
  'Cerda',
  'Cermak',
  'Chermak',
  'Cermak',
  'Cernochova',
  'Cernohous',
  'Cerny',
  'Cerney',
  'Cerny',
  'Cerv',
  'Cervenka',
  'Chalupka',
  'Charlott',
  'Chemlik',
  'Chicken',
  'Chilar',
  'Chromy',
  'Cihak',
  'Clineburg',
  'Klineberg',
  'Cober',
  'Colling',
  'Cvacek',
  'Czabal',
  'Damell',
  'Demall',
  'Dehmel',
  'Dana',
  'Dejmal',
  'Dempko',
  'Demko',
  'Dinko',
  'Divoky',
  'Dolejsi',
  'Dolezal',
  'Doljs',
  'Dopita',
  'Drassal',
  'Driml',
  'Duyava',

In [6]:
# 构建数据源NameClassDataset
class NameClassDataset(Dataset):
    def __init__(self, my_list_x, my_list_y):
        # 样本x
        self.my_list_x = my_list_x
        # 对应的标签y
        self.my_list_y = my_list_y
        self.sample_len = len(my_list_x)

    # 获取样本条数
    def __len__(self):
        return self.sample_len

    # 获取第几条样本数据
    def __getitem__(self, index):
        # 对index异常值进行修正 [0, self.sample_len-1]
        index = min(max(index, 0), self.sample_len - 1)

        x = self.my_list_x[index]
        y = self.my_list_y[index]

        """
        将样本x转换为one-hot编码
        以Alice为例：Alice可以看成是一个句子，每个字符都是一个分词
        将名字进行拆分 A - l - i - c - e
        每个字符都可以看成是一个one-hot编码
        总共有n_letters个字符，one-hot编码的维度为n_letters
        """
        # 形状为二维，第一维表示名字的所有字符向量，第二维表示每个字符的one-hot编码
        tensor_x = torch.zeros(len(x), n_letters)
        for index, char in enumerate(x):
            idx = all_letters.find(char)  # 字符在57个字符中的索引
            tensor_x[index][idx] = 1

        # 样本y 张量化
        tensor_y = torch.tensor(categorys.index(y), dtype=torch.long)
        return tensor_x, tensor_y

my_list_x, my_list_y = read_data("./assets/name_classfication.txt")
dataset = NameClassDataset(my_list_x, my_list_y)
print(dataset[0])
# mydataloader = DataLoader(dataset=dataset, batch_size=1, shuffle=True)

# for  i, (x, y) in enumerate (mydataloader):
#     print('x.shape', x.shape, x)
#     print('y.shape', y.shape, y)
#     break

['Abl\tCzech\n', 'Adsit\tCzech\n', 'Ajdrna\tCzech\n', 'Alt\tCzech\n', 'Antonowitsch\tCzech\n', 'Antonowitz\tCzech\n', 'Bacon\tCzech\n', 'Ballalatak\tCzech\n', 'Ballaltick\tCzech\n', 'Bartonova\tCzech\n', 'Bastl\tCzech\n', 'Baroch\tCzech\n', 'Benesch\tCzech\n', 'Betlach\tCzech\n', 'Biganska\tCzech\n', 'Bilek\tCzech\n', 'Blahut\tCzech\n', 'Blazek\tCzech\n', 'Blazek\tCzech\n', 'Blazejovsky\tCzech\n', 'Blecha\tCzech\n', 'Bleskan\tCzech\n', 'Blober\tCzech\n', 'Bock\tCzech\n', 'Bohac\tCzech\n', 'Bohunovsky\tCzech\n', 'Bolcar\tCzech\n', 'Borovka\tCzech\n', 'Borovski\tCzech\n', 'Borowski\tCzech\n', 'Borovsky\tCzech\n', 'Brabbery\tCzech\n', 'Brezovjak\tCzech\n', 'Brousil\tCzech\n', 'Bruckner\tCzech\n', 'Buchta\tCzech\n', 'Cablikova\tCzech\n', 'Camfrlova\tCzech\n', 'Cap\tCzech\n', 'Cerda\tCzech\n', 'Cermak\tCzech\n', 'Chermak\tCzech\n', 'Cermak\tCzech\n', 'Cernochova\tCzech\n', 'Cernohous\tCzech\n', 'Cerny\tCzech\n', 'Cerney\tCzech\n', 'Cerny\tCzech\n', 'Cerv\tCzech\n', 'Cervenka\tCzech\n', 'Cha